# Cooperative game theory — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def expected_payoff(A,p,q): return float(np.asarray(p) @ np.asarray(A) @ np.asarray(q))
def best_response_payoffs(A,q): return np.asarray(A) @ np.asarray(q)
print('setup ok')

## Shapley values as fair marginal contributions
Cooperative game theory asks how to divide coalition value. These examples compute a characteristic function, enumerate Shapley marginal contributions, test efficiency, show dummy and symmetry behavior, and compare to the core.

In [ ]:
# 1) A three-player characteristic function
players=['A','B','C']; v={():0,('A',):1,('B',):1,('C',):0,('A','B'):4,('A','C'):2,('B','C'):2,('A','B','C'):6}
vals=[v[tuple(s)] for r in range(4) for s in itertools.combinations(players,r)]
labels=[''.join(s) or '∅' for r in range(4) for s in itertools.combinations(players,r)]
plt.figure(figsize=(6,3)); plt.bar(labels,vals); plt.title('coalition values v(S)')
assert v[('A','B','C')]==6 and v[('A','B')]==4

In [ ]:
# 2) Shapley value by averaging marginal contributions over all orders
players=['A','B','C']; v={():0,('A',):1,('B',):1,('C',):0,('A','B'):4,('A','C'):2,('B','C'):2,('A','B','C'):6}
def val(S): return v[tuple(sorted(S))]
phi=dict.fromkeys(players,0.0); contribs={p:[] for p in players}
for order in itertools.permutations(players):
    S=[]
    for p in order:
        m=val(S+[p])-val(S); contribs[p].append(m); phi[p]+=m/6; S.append(p)
plt.figure(figsize=(5,3)); plt.bar(players,[phi[p] for p in players]); plt.title('Shapley values: A=2.5, B=2.5, C=1.0')
assert all(abs(phi[p]-e)<1e-12 for p,e in [('A',2.5),('B',2.5),('C',1.0)])

In [ ]:
# 3) Efficiency: Shapley allocations exactly exhaust grand-coalition value
vals=np.array([2.5,2.5,1.0]); cumulative=np.cumsum(vals)
plt.figure(figsize=(5,3)); plt.step(['A','B','C'],cumulative,where='mid'); plt.axhline(6,c='r',ls='--'); plt.ylim(0,6.5); plt.title('2.5 + 2.5 + 1.0 = 6')
assert abs(vals.sum()-6)<1e-12

In [ ]:
# 4) A dummy player receives exactly its stand-alone contribution
players=['A','B','D']; v={():0,('A',):2,('B',):0,('D',):1,('A','B'):5,('A','D'):3,('B','D'):1,('A','B','D'):6}
def val(S): return v[tuple(sorted(S))]
phi=dict.fromkeys(players,0.0)
for order in itertools.permutations(players):
    S=[]
    for p in order:
        phi[p]+=(val(S+[p])-val(S))/6; S.append(p)
plt.figure(figsize=(5,3)); plt.bar(players,[phi[p] for p in players]); plt.title('dummy D adds 1 to every coalition, gets 1')
assert abs(phi['D']-1.0)<1e-12

In [ ]:
# 5) Core constraints: no coalition wants to leave
alloc=np.array([2.5,2.5,1.0]); coal=[('A',),('B',),('C',),('A','B'),('A','C'),('B','C')]; req=np.array([1,1,0,4,2,2],dtype=float)
idx={'A':0,'B':1,'C':2}
paid=np.array([sum(alloc[idx[p]] for p in S) for S in coal])
plt.figure(figsize=(6,3)); plt.bar([''.join(S) for S in coal],paid-req); plt.axhline(0,c='k'); plt.title('slack >= 0 means the allocation is in the core')
assert np.all(paid>=req)